### Prospectivity Map

####  *AUTHOR:* Ehsan Farahbakhsh
####  *CONTACT:* e.farahbakhsh@sydney.edu.au
####  *DATE last modified:* 24/03/2026

This notebook uses the machine learning model trained in notebook `04_predictive_modelling` to analyse the probability of finding porphyry deposits within continental landmasses over time.

In addition to generating animations that visualise the temporal evolution of probability, it also allows users to create a global prospectivity map, 
along with corresponding uncertainty maps, for porphyry mineralisation.

We begin by importing the required libraries:

In [ ]:
# Import libraries
from ipywidgets import interact
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
import gplately
from gplately import EARTH_RADIUS, PlateModelManager
import joblib
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from sklearn.kernel_ridge import KernelRidge
from tqdm import tqdm

# Note: Ensure the 'lib' folder is located in the same folder as this notebook
from lib.prospectivity_map import *
from lib.feature_extraction import *
from lib.reconstruction_coregistration import *
from lib.predictive_modelling import *
from lib.plot import *

# Load configuration parameters (e.g., paths, model names)
from parameters import parameters

### Setup

As defined in `parameters.py`, the cell below configures the analysis parameters and specifies the paths to input/output files and directories. You can also specify the number of cores to use by setting an appropriate value for the `n_jobs` variable at the end of the cell.

**Note:** You can modify the analysis settings directly in `parameters.py`, located in the same folder as this notebook. The file is structured as a dictionary; look for keys such as `timespan` and `temporal_resolution` to adjust their values as needed.

In [ ]:
# Set up temporal analysis parameters
plate_model = parameters["plate_model"]
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"] # Youngest time to analyse
time_max = parameters["timespan"]["max"] # Oldest time to analyse
# Create an array of time steps for analysis (e.g., 0, 1, 2, ... Ma)
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

buffer_distance = parameters["buffer_distance"]
grid_resolution = parameters["grid_resolution"]
columns_to_drop_backarc = parameters["columns_to_drop_backarc"]
columns_to_drop_coastline = columns_to_drop_backarc.copy()

# Directory paths for inputs and outputs
inputs_dir = parameters["inputs_dir"]
outputs_dir = parameters["outputs_dir"]
figs_dir = parameters["figs_dir"]
buffer_zones_dir = parameters["buffer_zones_dir"]
coastlines_recon_dir = parameters["coastlines_recon_dir"]
ml_dir = parameters["ml_dir"]
coastline_prob_grids_dir = parameters["coastline_prob_grids_dir"]
coastline_entropy_grids_dir = parameters["coastline_entropy_grids_dir"]
coastline_vote_variance_grids_dir = parameters["coastline_vote_variance_grids_dir"]
coastline_prob_maps_dir = parameters["coastline_prob_maps_dir"]
coastline_entropy_maps_dir = parameters["coastline_entropy_maps_dir"]
coastline_vote_variance_maps_dir = parameters["coastline_vote_variance_maps_dir"]

# Data filenames
subduction_data_filename = parameters["subduction_data_filename"]
coastlines_filename = parameters["coastlines_filename"]
coastlines_recon_filename = parameters["coastlines_recon_filename"]
coastline_coords_filename = parameters["coastline_coords_filename"]
coastline_data_filename = parameters["coastline_data_filename"]
coastline_prob_filename = parameters["coastline_prob_filename"]
deposit_coords_recon_all_filtered_filename = parameters["deposit_coords_recon_all_filtered_filename"]
coastline_prob_present_day_filename = parameters["coastline_prob_present_day_filename"]
coastline_prob_present_day_grid_filename = parameters["coastline_prob_present_day_grid_filename"]
coastline_entropy_present_day_grid_filename = parameters["coastline_entropy_present_day_grid_filename"]
coastline_vote_variance_present_day_grid_filename = parameters["coastline_vote_variance_present_day_grid_filename"]
coastline_uncertainty_present_day_grid_filename = parameters["coastline_uncertainty_present_day_grid_filename"]
coastline_prob_unc_present_day_grid_filename = parameters["coastline_prob_unc_present_day_grid_filename"]
deposit_coords_recon_filename = parameters["deposit_coords_recon_filename"]

model_rf_calibrated_filename = parameters["model_rf_calibrated_filename"]
robust_scaler_filename = parameters["robust_scaler_filename"]

# Construct full directory paths
figs_dir = os.path.join(outputs_dir, figs_dir)
ml_dir = os.path.join(outputs_dir, ml_dir)
buffer_zones_dir = os.path.join(outputs_dir, buffer_zones_dir)
coastlines_recon_dir = os.path.join(outputs_dir, coastlines_recon_dir)
coastline_prob_grids_dir = os.path.join(ml_dir, coastline_prob_grids_dir)
coastline_entropy_grids_dir = os.path.join(ml_dir, coastline_entropy_grids_dir)
coastline_vote_variance_grids_dir = os.path.join(ml_dir, coastline_vote_variance_grids_dir)
coastline_prob_maps_dir = os.path.join(ml_dir, coastline_prob_maps_dir)
coastline_entropy_maps_dir = os.path.join(ml_dir, coastline_entropy_maps_dir)
coastline_vote_variance_maps_dir = os.path.join(ml_dir, coastline_vote_variance_maps_dir)

# Construct full file paths
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)
coastlines_filename = os.path.join(inputs_dir, coastlines_filename)
coastlines_recon_filename = os.path.join(plate_model, "Coastlines", coastlines_recon_filename)
coastline_coords_filename = os.path.join(outputs_dir, coastline_coords_filename)
coastline_data_filename = os.path.join(outputs_dir, coastline_data_filename)
coastline_prob_filename = os.path.join(ml_dir, coastline_prob_filename)
deposit_coords_recon_all_filtered_filename = os.path.join(outputs_dir, deposit_coords_recon_all_filtered_filename)
coastline_prob_present_day_filename = os.path.join(ml_dir, coastline_prob_present_day_filename)
coastline_prob_present_day_grid_filename = os.path.join(ml_dir, coastline_prob_present_day_grid_filename)
coastline_entropy_present_day_grid_filename = os.path.join(ml_dir, coastline_entropy_present_day_grid_filename)
coastline_vote_variance_present_day_grid_filename = os.path.join(ml_dir, coastline_vote_variance_present_day_grid_filename)
coastline_uncertainty_present_day_grid_filename = os.path.join(ml_dir, coastline_uncertainty_present_day_grid_filename)
coastline_prob_unc_present_day_grid_filename = os.path.join(ml_dir, coastline_prob_unc_present_day_grid_filename)
deposit_coords_recon_filename = os.path.join(outputs_dir, deposit_coords_recon_filename)

model_rf_calibrated_filename = os.path.join(ml_dir, model_rf_calibrated_filename)
robust_scaler_filename = os.path.join(ml_dir, robust_scaler_filename)

# Paths to different types of grids
agegrid_dir = os.path.join(inputs_dir, "SeafloorAge")
crusthick_dir = os.path.join(inputs_dir, "CrustalThickness")

# Number of cores to be used for running this notebook
n_jobs = 20

The cell below loads the necessary files to create the `PlateReconstruction` and `PlateTopologies` objects, which will later be used for reconstruction and generating visualisations.

In [ ]:
# Plate motion model
pmm = PlateModelManager()
pm = pmm.get_model(plate_model)

rotation_model = pm.get_rotation_model()
topology_features = pm.get_topologies()

static_polygons = pm.get_static_polygons()
coastlines = pm.get_coastlines()
continents = pm.get_continental_polygons()
COBs = pm.get_COBs()

plate_reconstruction = PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=topology_features,
    static_polygons=static_polygons,
)

gplot = PlotTopologies(
    plate_reconstruction=plate_reconstruction,
    coastlines=coastlines,
    continents=continents,
    COBs=COBs,
)

# Define map projection (Mollweide provides a good global view) and central meridian
projection = ccrs.Mollweide(central_longitude=60)
central_meridian = 0

### Points within Continental Landmasses

To generate a single global prospectivity map showing mineralisation probability at present-day coordinates using the trained model, we must reconstruct the points within the present-day continental landmass and track how their probabilities change over time. The process begins by generating a regular grid of points within the present-day continental landmass and reconstructing their locations over time using a plate motion model, retaining only those that fall within the buffer zones representing arc–backarc environments. Running the cell below results in a dataframe containing age and the present-day and reconstructed latitude and longitude of the points across the continental landmasses through time.

In [ ]:
if os.path.isfile(coastline_coords_filename):
    coastline_coords = pd.read_csv(coastline_coords_filename)
else:
    # Reconstruct the points within the present-day continental landmass through time
    coastline_coords = reconstruct_coastline_nodes(
        coastlines=coastlines_filename, # Present-day coastlines
        coastlines_recon=coastlines_recon_filename, # Present-day reconstructable coastlines
        resolution=grid_resolution, # The distance between the points
        buffer_zones_dir=buffer_zones_dir,
        time_steps=time_steps,
        plate_reconstruction=plate_reconstruction,
        coastlines_recon_dir = coastlines_recon_dir,
        output_filename=coastline_coords_filename,
        verbose=True,
    )

This section generates an interactive map that displays the reconstructed points together with seafloor age and plate boundaries—including mid-ocean ridges and transform faults—at a selected time.

In [ ]:
@interact
def show_map(time=time_steps):
    gplot.time = time    
    
    # Load seafloor age grid for the specified geological time
    agegrid_filename =  f"seafloor_age_{time}Ma.nc"
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba("darkgray", alpha=0.5))
    )

    # Plot seafloor age as background
    im = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)
    # Plot continental areas in dark grey
    gplot.plot_coastlines(ax, facecolor="darkgray", edgecolor="none", zorder=2)
    # Plot plate motion vectors (arrows showing plate movement direction)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # Load and plot buffer zones for the specified geological time
    # buffer_zones_t_filename = f"buffer_zones_{time}Ma.geojson"
    # buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    # buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    # buffer_zones_t.plot(
    #     ax=ax,
    #     transform=ccrs.PlateCarree(),
    #     facecolor="palegreen",
    #     edgecolor="none",
    #     alpha=0.7,
    #     zorder=4,
    # )

    # Plot grid points
    coastline_coords_t = coastline_coords[coastline_coords["age (Ma)"] == time]

    gplot.plot_topological_plate_boundaries(ax, color="dimgray", linewidth=1.2, zorder=5)

    ax.scatter(
        coastline_coords_t["lon"],
        coastline_coords_t["lat"],
        transform=ccrs.PlateCarree(),
        marker=".",
        c="red",
        s=1,
        alpha=0.5,
        zorder=6
    )
    
    gplot.plot_trenches(ax, color="black", alpha=0.5, zorder=7)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color="black", alpha=0.5, zorder=8)

    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=9)

    ax.text(0.49,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, "60°W", transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
    
    cb = fig.colorbar(im, orientation="horizontal", shrink=0.4, pad=0.06, extend="max")
    cb.set_label("Seafloor Age (Myr)", fontsize=16, labelpad=10)
    cb.set_ticks([0, 50, 100, 150, 200])
    cb.ax.tick_params(labelsize=16)

    # Dummy handle to trigger custom handler
    trench_handle = Line2D([], [], color="black")
    
    # Add custom handles
    custom_handles = [
        Patch(facecolor="darkgray", edgecolor="none"),
        # Patch(facecolor="palegreen", edgecolor="none", alpha=0.7),
        trench_handle,
        Line2D([0], [0], color="dimgray", lw=1.2),
        Line2D([0], [0], marker=".", markerfacecolor="red", markeredgecolor="none", markersize=10, linestyle="None")
    ]
    custom_labels = [
        "Continental crust",
        # "Target arc environment",
        "Subduction zone",
        "Plate boundary",
        "Target point within\narc environment and\ncontinental landmass",
    ]

    # Draw legend
    ax.legend(custom_handles, custom_labels, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.3),
              handler_map={trench_handle: HandlerTrenchLine()})

    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
        
    plt.show()

### Coregistration

This code block performs co-registration by finding the closest trench point to each point in the previously created point set within the buffer zones. It then assigns the feature values of the nearest trench point to each of the points, along with the distance between them. The `run_coregister_crustal_thickness` function is specifically designed to coregister points with crustal thickness grids. In the final step, missing values within the set of points are handled using the `fill_missing_values` function, which supports two imputation strategies: iterative imputation and median-value imputation. Because iterative imputation is computationally expensive, we use median values here.

In [ ]:
if os.path.isfile(coastline_data_filename):
    coastline_data = pd.read_csv(coastline_data_filename)
else:
    # The feature values calculated using the "01_feature_extraction" notebook will be loaded
    subduction_data = pd.read_csv(subduction_data_filename)

    coastline_data = run_coregister_point_data(
        point_data=coastline_coords,
        subduction_data=subduction_data,
        keep_index=True,
        n_jobs=n_jobs,
        verbose=True,
    )
    
    coastline_data = fill_missing_values(
        coastline_data,
        method="median",
        exclude_columns=columns_to_drop_coastline,
        # distance_threshold=distance_threshold,
        report_missing=True,
        random_state=42,
        export_to_csv=True,
        output_path=coastline_data_filename,
        n_jobs=n_jobs,
        verbose=True
    )

### Calculate Probability, Entropy, and Vote Variance

We apply the trained model in notebook `04_predictive_modelling` to the previously generated points located within buffer zones that encompass arc–backarc environments, in order to estimate:

- **Mineralisation Probability**: The model's confidence that geological conditions are favourable for deposit formation.  
- **Entropy**: A measure of prediction uncertainty (high entropy indicates greater model uncertainty).  
- **Vote Variance**: The variability among decision trees (high variance suggests conflicting results between trees).  

The resulting predictions are saved as global grids, highlighting where mineralisation is most likely and where the model is more or less confident. These grids make it possible to visualise variations in mineralisation probability, helping to identify both potential exploration targets and areas of uncertainty in the predictions.

In [ ]:
if os.path.isfile(coastline_prob_filename):
    print(f"Probabilities have already been calculated and saved in {coastline_prob_filename}\nLoading ...")
    coastline_prob = pd.read_csv(coastline_prob_filename)
else:
    # Load model and scaler
    model_rf_calibrated = joblib.load(model_rf_calibrated_filename)
    model_rf_calibrated.n_jobs = n_jobs
    robust_scaler = joblib.load(robust_scaler_filename)

    if "valid" in coastline_data.columns:
        # Filter valid rows
        coastline_data = coastline_data[coastline_data["valid"]].copy()

    # Scale feature values using the same scaler as training
    selected_features = model_rf_calibrated.feature_names_in_
    coastline_scaled = robust_scaler.transform(coastline_data[selected_features])
    coastline_scaled = pd.DataFrame(coastline_scaled, columns=selected_features)

    # Calculate probability and entropy values for all points
    # Entropy quantifies uncertainty in model predictions
    probs = model_rf_calibrated.predict_proba(coastline_scaled)[:, 1].ravel()

    # Prepare final DataFrame
    columns_prob = [
        "index",
        "lon",
        "lat",
        "present_lon",
        "present_lat",
        "age (Ma)",
    ]
    
    coastline_prob = coastline_data[columns_prob].copy()
    coastline_prob["probability"] = probs

    entropies = calculate_entropy(probs)
    coastline_prob["entropy"] = entropies
    
    # Calculate vote variance
    # Vote variance shows disagreement among decision trees
    chunk_size = 100000  # Adjust based on available memory
    n_samples = coastline_scaled.shape[0]
    n_chunks = (n_samples // chunk_size) + (1 if n_samples % chunk_size != 0 else 0)  # Ensure last chunk is included
    
    for i in tqdm(range(n_chunks)):
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, n_samples)
    
        X_chunk = coastline_scaled.iloc[start_idx:end_idx]
        
        # Calculate vote variance for the chunk
        vote_variances_chunk = calculate_tree_vote_variance(
            model_rf_calibrated.estimator,
            X_chunk.values
        )
        
        # Ensure that the chunk size matches the number of rows being processed
        if len(vote_variances_chunk) != (end_idx - start_idx):
            raise ValueError(f"Mismatch in size: {len(vote_variances_chunk)} vs. {end_idx - start_idx}")
        
        # Assign the result to the appropriate slice of the dataframe
        coastline_prob.loc[start_idx:end_idx-1, "vote_variance"] = vote_variances_chunk

    # Normalise entropy and vote variance to [0, 1]
    entropy_min = coastline_prob["entropy"].min()
    entropy_max = coastline_prob["entropy"].max()
    variance_min = coastline_prob["vote_variance"].min()
    variance_max = coastline_prob["vote_variance"].max()

    coastline_prob["entropy_norm"] = (
        (coastline_prob["entropy"] - entropy_min) / (entropy_max - entropy_min)
        if entropy_max > entropy_min else 0.0
    )

    coastline_prob["vote_variance_norm"] = (
        (coastline_prob["vote_variance"] - variance_min) / (variance_max - variance_min)
        if variance_max > variance_min else 0.0
    )

    # Combined uncertainty
    coastline_prob["uncertainty"] = (
        0.5 * coastline_prob["entropy_norm"]
        + 0.5 * coastline_prob["vote_variance_norm"]
    )

    # Probability adjusted by uncertainty
    coastline_prob["probability_uncertainty"] = (
        coastline_prob["probability"] * (1 - coastline_prob["uncertainty"])
    )
    
    # Export rows with calculated metrics
    print(f"Saving the metrics as a CSV file: {coastline_prob_filename}")
    coastline_prob.to_csv(coastline_prob_filename, index=False)

# Create probability grids
if os.path.exists(coastline_prob_grids_dir):
    print(f"Probability grids are located in {coastline_prob_grids_dir}")
else:
    os.makedirs(coastline_prob_grids_dir, exist_ok=True)

    create_grids(
        data=coastline_prob_filename,
        output_dir=coastline_prob_grids_dir,
        times=time_steps,
        resolution=grid_resolution,
        extent=(-180, 180, -90, 90),
        interpolation=True,
        threads=n_jobs,
        verbose=True,
        column="probability",
        filename_format="coastline_probability_grid_{}Ma.nc",
    )

# Create entropy grids
if os.path.exists(coastline_entropy_grids_dir):
    print(f"Entropy grids are located in {coastline_entropy_grids_dir}")
else:
    os.makedirs(coastline_entropy_grids_dir, exist_ok=True)

    create_grids(
        data=coastline_prob_filename,
        output_dir=coastline_entropy_grids_dir,
        times=time_steps,
        resolution=grid_resolution,
        extent=(-180, 180, -90, 90),
        interpolation=True,
        threads=n_jobs,
        verbose=True,
        column="entropy",
        filename_format="coastline_entropy_grid_{}Ma.nc",
    )

# Create vote variance grids
if os.path.exists(coastline_vote_variance_grids_dir):
    print(f"Vote variance grids are located in {coastline_vote_variance_grids_dir}")
else:
    os.makedirs(coastline_vote_variance_grids_dir, exist_ok=True)

    create_grids(
        data=coastline_prob_filename,
        output_dir=coastline_vote_variance_grids_dir,
        times=time_steps,
        resolution=grid_resolution,
        extent=(-180, 180, -90, 90),
        interpolation=True,
        threads=n_jobs,
        verbose=True,
        column="vote_variance",
        filename_format="coastline_vote_variance_grid_{}Ma.nc",        
    )

### Probability Maps

This section generates an interactive probability map that displays mineralisation probability along trench lines within the continental landmasses, alongside known mineral occurrences. It also plots seafloor age and plate boundaries, including mid-ocean ridges and transform faults, at a selected time. The visualisation highlights spatial patterns, showing how mineralisation probability relates to tectonic features, and temporal evolution, showing how favourable conditions migrate through time.

In [ ]:
@interact
def show_map(time=time_steps):
    gplot.time = time # Set reconstruction time
    fig = plot(
        gplot=gplot,
        grid=os.path.join(
            coastline_prob_grids_dir,
            f"coastline_probability_grid_{time:0.0f}Ma.nc", # Mineralisation probability grid
        ),
        grid_type="probability",
        background_grid=os.path.join(
            agegrid_dir,
            f"seafloor_age_{time:0.0f}Ma.nc", # Seafloor age grid
        ),
        positives=deposit_coords_recon_all_filtered_filename, # Reconstructed location of known mineral occurrences
        projection=projection,
        time=time,
        central_meridian=central_meridian,
    )

The cell below generates a full sequence of mineralisation probability maps through geological time and compiles them into a time-lapse animation, illustrating how the modelled probability of mineralisation changes over time.

In [ ]:
# Generate all probability maps and create animation
if os.path.exists(coastline_prob_maps_dir):
    print(f"Probability maps are located in {coastline_prob_maps_dir}")
else:
    os.makedirs(coastline_prob_maps_dir, exist_ok=True)

    # Create maps in parallel for efficiency
    with Parallel(
        n_jobs=n_jobs,
        verbose=True,
        pre_dispatch="all",
        batch_size=int(len(time_steps) // n_jobs) + 1,
    ) as parallel:

        # Define output filename for each time step
        output_filenames = [
            os.path.join(coastline_prob_maps_dir, f"coastline_probability_map_{t:0.0f}Ma.png")
            for t in time_steps
        ]

        parallel(
            delayed(plot_parallel)(
                rotation_model,
                topology_features,
                static_polygons,
                coastlines,
                continents,
                COBs,
                grid=os.path.join(
                    coastline_prob_grids_dir,
                    f"coastline_probability_grid_{t:0.0f}Ma.nc",
                ),
                grid_type="probability",
                background_grid=os.path.join(
                    agegrid_dir,
                    f"seafloor_age_{t:0.0f}Ma.nc",
                ),
                positives=deposit_coords_recon_all_filtered_filename,
                projection=projection,
                time=t,
                output_filename=o,
                central_meridian=central_meridian,
            )
            for t, o in zip(time_steps, output_filenames)
        )

    # Create an animation showing the evolution of mineralisation probability
    # Animation reveals temporal patterns in geological processes
    output_filename = os.path.join(ml_dir, "coastline_probability_animation.mp4")
    create_animation(
        image_filenames=output_filenames[::-1],
        output_filename=output_filename,
        fps=10,
        bitrate="5000k",
    )

### Entropy Maps

This section generates an interactive entropy map along trench lines within the continental landmasses, combined with known mineral occurrences. It also displays seafloor age and plate boundaries—including mid-ocean ridges and transform faults—at a chosen time step.

Entropy is a statistical measure of uncertainty in the model's probability predictions:

- **Low entropy (≈ 0):** The model is confident. Predictions lean strongly toward either a high or low probability of mineralisation.
- **High entropy (≈ 1):** The model is uncertain. Probabilities cluster near 50%, indicating ambiguity about whether conditions favour or hinder deposit formation.

Entropy maps allow you to explore how spatial uncertainty in mineralisation changes through geological time. By sliding across different time steps, you can visualise regions where the model's predictions are most confident versus where they remain uncertain.

In [ ]:
@interact
def show_map(time=time_steps):
    gplot.time = time # Set reconstruction time
    fig = plot(
        gplot=gplot,
        grid=os.path.join(
            coastline_entropy_grids_dir,
            f"coastline_entropy_grid_{time:0.0f}Ma.nc", # Entropy grid
        ),
        grid_type="entropy",
        background_grid=os.path.join(
            agegrid_dir,
            f"seafloor_age_{time:0.0f}Ma.nc", # Seafloor age grid
        ),
        positives=deposit_coords_recon_all_filtered_filename, # Reconstructed location of known mineral occurrences
        projection=projection,
        time=time,
        central_meridian=central_meridian,
    )

The cell below generates a complete sequence of entropy maps through geological time and compiles them into a time-lapse animation. This visualisation shows how entropy evolves over time, providing an intuitive way to communicate model uncertainty across Earth's history. Such animations are especially useful for identifying when and where the predictive model is most stable or most variable.

In [ ]:
# Generate all entropy maps and create animation
if os.path.exists(coastline_entropy_maps_dir):
    print(f"Entropy maps are located in {coastline_entropy_maps_dir}")
else:
    os.makedirs(coastline_entropy_maps_dir, exist_ok=True)

    # Create maps in parallel for efficiency
    with Parallel(
        n_jobs=n_jobs,
        verbose=True,
        pre_dispatch="all",
        batch_size=int(len(time_steps) // n_jobs) + 1,
    ) as parallel:

        # Define output filename for each time step
        output_filenames = [
            os.path.join(coastline_entropy_maps_dir, f"coastline_entropy_map_{t:0.0f}Ma.png")
            for t in time_steps
        ]

        parallel(
            delayed(plot_parallel)(
                rotation_model,
                topology_features,
                static_polygons,
                coastlines,
                continents,
                COBs,
                grid=os.path.join(
                    coastline_entropy_grids_dir,
                    f"coastline_entropy_grid_{t:0.0f}Ma.nc",
                ),
                grid_type="entropy",
                background_grid=os.path.join(
                    agegrid_dir,
                    f"seafloor_age_{t:0.0f}Ma.nc",
                ),
                positives=deposit_coords_recon_all_filtered_filename,
                projection=projection,
                time=t,
                output_filename=o,
                central_meridian=central_meridian,
            )
            for t, o in zip(time_steps, output_filenames)
        )

    # Create an animation showing the changes in entropy through geological time
    output_filename = os.path.join(ml_dir, "coastline_entropy_animation.mp4")
    create_animation(
        image_filenames=output_filenames[::-1],
        output_filename=output_filename,
        fps=10,
        bitrate="5000k",
    )

### Vote Variance Maps

Vote variance maps illustrate model disagreement—showing where different iterations of a predictive model produce varying estimates of mineralisation likelihood. Each tree in the Random Forest casts a vote for a given location, predicting either the presence, absence, or probability of mineralisation. The vote variance is then calculated as the statistical variance of these votes across all trees.

High vote variance indicates less agreement between models, highlighting uncertainty in the geological features driving mineralisation. Conversely, low variance reflects model consensus and suggests more stable predictive signals.

The following code segment generates an interactive plot of a vote variance map for a selected geological time, enabling users to explore model agreement and disagreement in predicted mineral prospectivity through time.

In [ ]:
@interact
def show_map(time=time_steps):
    gplot.time = time # Set reconstruction time
    fig = plot(
        gplot=gplot,
        grid=os.path.join(
            coastline_vote_variance_grids_dir,
            f"coastline_vote_variance_grid_{time:0.0f}Ma.nc", # Vote variance grid
        ),
        grid_type="vote variance",
        background_grid=os.path.join(
            agegrid_dir,
            f"seafloor_age_{time:0.0f}Ma.nc", # Seafloor age grid
        ),
        positives=deposit_coords_recon_all_filtered_filename, # Reconstructed location of known mineral occurrences
        projection=projection,
        time=time,
        central_meridian=central_meridian,
    )

The cell below generates a full sequence of vote variance maps across geological time and compiles them into a time-lapse animation. This visualisation highlights where model consensus on mineralisation is strong or weak at each time step.

In [ ]:
# Generate all vote variance maps and create animation
if os.path.exists(coastline_vote_variance_maps_dir):
    print(f"Vote variance maps are located in {coastline_vote_variance_maps_dir}")
else:
    os.makedirs(coastline_vote_variance_maps_dir, exist_ok=True)

    # Create maps in parallel for efficiency
    with Parallel(
        n_jobs=20,
        verbose=True,
        pre_dispatch="all",
        batch_size=int(len(time_steps) // n_jobs) + 1,
    ) as parallel:

        # Define output filename for each time step
        output_filenames = [
            os.path.join(coastline_vote_variance_maps_dir, f"coastline_vote_variance_map_{t:0.0f}Ma.png")
            for t in time_steps
        ]

        parallel(
            delayed(plot_parallel)(
                rotation_model,
                topology_features,
                static_polygons,
                coastlines,
                continents,
                COBs,
                grid=os.path.join(
                    coastline_vote_variance_grids_dir,
                    f"coastline_vote_variance_grid_{t:0.0f}Ma.nc",
                ),
                grid_type="vote variance",
                background_grid=os.path.join(
                    agegrid_dir,
                    f"seafloor_age_{t:0.0f}Ma.nc",
                ),
                positives=deposit_coords_recon_all_filtered_filename,
                projection=projection,
                time=t,
                output_filename=o,
                central_meridian=central_meridian,
            )
            for t, o in zip(time_steps, output_filenames)
        )

    # Create an animation showing the changes in vote variance through geological time
    output_filename = os.path.join(ml_dir, "coastline_vote_variance_animation.mp4")
    create_animation(
        image_filenames=output_filenames[::-1],
        output_filename=output_filename,
        fps=10,
        bitrate="5000k",
    )

### Trend of Average Probability, Entropy, and Vote Variance

Here, we calculate the average probability values greater than 0.5 for each of the points generated at the beginning of this notebook over time. By filtering out low-probability predictions (probability ≤ 0.5), we focus on promising sites, resulting in a clean dataset that highlights the most probable and geologically reliable locations for porphyry mineralisation at present-day coordinates.

In [ ]:
if os.path.isfile(coastline_prob_present_day_filename):
    coastline_prob_present_day = pd.read_csv(coastline_prob_present_day_filename)
else:
    # Filter for high-probability locations
    coastline_prob_high = coastline_prob[coastline_prob["probability"] > 0.5]
    # Average probability, entropy, vote variance, uncertainty, and the integration of probability and uncertainty for each unique location index
    coastline_prob_high_mean = coastline_prob_high.groupby("index")[["probability", "entropy", "vote_variance", "uncertainty", "probability_uncertainty"]].mean().reset_index()
    present_day_coords = coastline_data[["index", "present_lon", "present_lat"]].drop_duplicates(subset="index")
    # Merge the averaged probabilities with present-day coordinates
    coastline_prob_present_day = pd.merge(coastline_prob_high_mean, present_day_coords, on="index", how="left")
    coastline_prob_present_day = coastline_prob_present_day[["index", "present_lon", "present_lat", "probability", "entropy", "vote_variance", "uncertainty", "probability_uncertainty"]]
    # Save output
    coastline_prob_present_day.to_csv(coastline_prob_present_day_filename, index=False)

This section visualises how the average predicted probability of porphyry mineralisation for points within the continental landmasses varies over geological time, and relates these probabilities to both the number of known deposits at different ages (shown as a histogram) and the average value of a selected feature associated with those probabilities. This highlights how the feature evolves through time.

It also provides the option to plot the uncertainty envelope for the average trends by calculating the standard deviation of values at each time step. The curves are smoothed to emphasise trends, and dual y-axes enable direct comparison between probabilities, deposit frequency, and feature values, providing a comprehensive temporal overview. In doing so, the visualisation reveals how the predicted probability of mineralisation changes across Earth's history, helping to identify key time periods favourable for porphyry formation.

In [ ]:
# Spread function
def compute_spread(x, method="std", scale=False):
    x = pd.Series(x).dropna()

    if len(x) == 0:
        return np.nan

    if method == "std":
        return x.std()
    elif method == "mad":
        mad = np.median(np.abs(x - np.median(x)))
        return mad * 1.4826 if scale else mad
    else:
        raise ValueError("method must be 'std' or 'mad'")
        
model_rf_calibrated = joblib.load(model_rf_calibrated_filename)
selected_features = model_rf_calibrated.feature_names_in_

deposit_coords_recon = pd.read_csv(deposit_coords_recon_filename)

# coastline_data = pd.read_csv(coastline_data_filename)
distance_threshold = np.deg2rad(buffer_distance) * EARTH_RADIUS
coastline_data_filtered = coastline_data[
    coastline_data["distance_to_trench (km)"] <= distance_threshold
]

bin_width = 20
bins = np.arange(time_min, time_max + bin_width, bin_width)

@interact
def show_map(
    feature=selected_features,
    spread_method=["std", "mad"],
    probability_col=["probability", "probability_uncertainty"],
):
    coastline_feat_age = coastline_prob.merge(
        coastline_data_filtered[["index", "age (Ma)", feature]],
        on=["index", "age (Ma)"],
        how="left"
    )

    # Probability mean by age
    coastline_prob_mean_age = (
        coastline_prob.groupby("age (Ma)")[probability_col]
        .mean()
        .reset_index()
        .sort_values("age (Ma)")
    )

    ages = coastline_prob_mean_age["age (Ma)"].values.reshape(-1, 1)
    probs = coastline_prob_mean_age[probability_col].values.reshape(-1, 1)

    # Smooth mean curve
    ages_smooth, probs_smooth = smooth_curve(
        ages, probs, method="gaussian", fwhm_myr=50
    )

    # Probability spread by age
    coastline_prob_spread_age = (
        coastline_prob.groupby("age (Ma)")[probability_col]
        .apply(lambda x: compute_spread(x, method=spread_method, scale=False))
        .reset_index(name=probability_col)
        .sort_values("age (Ma)")
    )

    probs_spread = coastline_prob_spread_age[probability_col].values.reshape(-1, 1)
    _, probs_spread_smooth = smooth_curve(
        ages, probs_spread, method="gaussian", fwhm_myr=50
    )

    # Feature mean + spread by age
    coastline_feat_mean_age = (
        coastline_feat_age.groupby("age (Ma)")[feature]
        .mean()
        .reset_index()
        .sort_values("age (Ma)")
    )

    coastline_feat_spread_age = (
        coastline_feat_age.groupby("age (Ma)")[feature]
        .apply(lambda x: compute_spread(x, method=spread_method, scale=False))
        .reset_index(name=feature)
        .sort_values("age (Ma)")
    )

    feats = coastline_feat_mean_age[feature].values.reshape(-1, 1)
    feats_spread = coastline_feat_spread_age[feature].values.reshape(-1, 1)

    _, feats_smooth = smooth_curve(
        ages, feats, method="gaussian", fwhm_myr=50
    )
    _, feats_spread_smooth = smooth_curve(
        ages, feats_spread, method="gaussian", fwhm_myr=50
    )

    # Plot
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_facecolor("whitesmoke")

    # Histogram
    ax1.hist(
        deposit_coords_recon["age (Ma)"],
        bins=bins,
        orientation="vertical",
        color="gray",
        alpha=0.5,
        edgecolor="black",
        zorder=1
    )
    ax1.set_xlabel("Age (Ma)", fontsize=14)
    ax1.set_ylabel("Deposit count", fontsize=14)

    # Probability axis
    ax2 = ax1.twinx()
    prob_label = (
        "Mineralisation probability"
        if probability_col == "probability"
        else "Mineralisation probability adjusted by uncertainty"
    )

    ax2.plot(
        ages_smooth,
        probs_smooth,
        color="orangered",
        linewidth=2,
        label=prob_label,
        zorder=2
    )
    ax2.fill_between(
        ages_smooth.flatten(),
        (probs_smooth - probs_spread_smooth).flatten(),
        (probs_smooth + probs_spread_smooth).flatten(),
        color="orangered",
        alpha=0.2
    )

    ax2.set_xlim(time_max, time_min)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel(
        "Mineralisation probability"
        if probability_col == "probability"
        else "Mineralisation probability adjusted by uncertainty",
        fontsize=14
    )

    # Feature axis
    ax3 = ax1.twinx()
    ax3.spines["right"].set_position(("outward", 60))

    feature_label = format_feature_name(feature)
    uncertainty_label = "±1σ" if spread_method == "std" else "±MAD"

    ax3.plot(
        ages_smooth,
        feats_smooth,
        color="royalblue",
        linewidth=2,
        label=f"{feature_label} ({uncertainty_label})",
        zorder=2
    )
    ax3.fill_between(
        ages_smooth.flatten(),
        (feats_smooth - feats_spread_smooth).flatten(),
        (feats_smooth + feats_spread_smooth).flatten(),
        color="royalblue",
        alpha=0.2
    )

    ax3.set_ylabel(feature_label, fontsize=14)

    ax1.tick_params(labelsize=12)
    ax2.tick_params(labelsize=12)
    ax3.tick_params(labelsize=12)

    # Legend
    spread_label = "±1σ" if spread_method == "std" else "±MAD"

    lines2, labels2 = ax2.get_legend_handles_labels()
    lines3, labels3 = ax3.get_legend_handles_labels()

    labels2 = [f"{l} ({spread_label})" for l in labels2]

    lines = lines2 + lines3
    labels = labels2 + labels3
    ax3.legend(lines, labels, loc="upper left", frameon=True, fontsize=12)

    plt.tight_layout()

    clean_feature = clean_feature_name(feature)
    fig_path = os.path.join(figs_dir, f"coastline_avg_trend_{clean_feature}.png")
    if not os.path.exists(fig_path):
        plt.savefig(
            fig_path,
            dpi=300,
            bbox_inches="tight",
        )
    
    plt.show()

This cell is similar to the previous one, but instead of averaging all probability values calculated for a point, it averages only those greater than 0.5.

In [ ]:
@interact(
    feature=selected_features,
    spread_method=["std", "mad"],
    probability_col=["probability", "probability_uncertainty"],
)
def show_map(feature, spread_method="std", probability_col="probability"):

    coastline_prob_high = coastline_prob[coastline_prob[probability_col] > 0.5]

    coastline_prob_high_mean_age = (
        coastline_prob_high.groupby("age (Ma)")[probability_col]
        .mean()
        .reset_index()
    )

    # Get all unique ages from original data
    all_ages = coastline_prob["age (Ma)"].unique()
    complete_ages = pd.DataFrame({"age (Ma)": all_ages})

    # Merge and fill missing values
    coastline_prob_high_mean_age = complete_ages.merge(
        coastline_prob_high_mean_age,
        on="age (Ma)",
        how="left"
    ).fillna(coastline_prob_high_mean_age[probability_col].mean())

    # Sort by age
    coastline_prob_high_mean_age = coastline_prob_high_mean_age.sort_values("age (Ma)")

    ages_high = coastline_prob_high_mean_age["age (Ma)"].values.reshape(-1, 1)
    probs_high = coastline_prob_high_mean_age[probability_col].values.reshape(-1, 1)

    # Smooth the probability mean once
    ages_high_smooth, probs_high_smooth = smooth_curve(
        ages_high, probs_high, method="gaussian", fwhm_myr=50
    )

    # Probability spread by age
    coastline_prob_high_spread_age = (
        coastline_prob_high.groupby("age (Ma)")[probability_col]
        .apply(lambda x: compute_spread(x, method=spread_method))
        .reset_index(name=probability_col)
    )

    coastline_prob_high_spread_age = complete_ages.merge(
        coastline_prob_high_spread_age,
        on="age (Ma)",
        how="left"
    ).fillna(coastline_prob_high_spread_age[probability_col].mean())

    coastline_prob_high_spread_age = coastline_prob_high_spread_age.sort_values("age (Ma)")
    probs_high_spread = coastline_prob_high_spread_age[probability_col].values.reshape(-1, 1)

    _, probs_high_spread_smooth = smooth_curve(
        ages_high, probs_high_spread, method="gaussian", fwhm_myr=50
    )

    # Merge feature data for high-probability subset
    coastline_feat_high = coastline_prob_high.merge(
        coastline_data_filtered[["index", "age (Ma)", feature]],
        on=["index", "age (Ma)"],
        how="left"
    )

    # Feature mean by age
    coastline_feat_high_mean_age = (
        coastline_feat_high.groupby("age (Ma)")[feature]
        .mean()
        .reset_index()
    )

    coastline_feat_high_mean_age = complete_ages.merge(
        coastline_feat_high_mean_age,
        on="age (Ma)",
        how="left"
    ).fillna(coastline_feat_high_mean_age[feature].mean())

    # Feature spread by age
    coastline_feat_high_spread_age = (
        coastline_feat_high.groupby("age (Ma)")[feature]
        .apply(lambda x: compute_spread(x, method=spread_method))
        .reset_index(name=feature)
    )

    coastline_feat_high_spread_age = complete_ages.merge(
        coastline_feat_high_spread_age,
        on="age (Ma)",
        how="left"
    ).fillna(coastline_feat_high_spread_age[feature].mean())

    # Sort by age
    coastline_feat_high_mean_age = coastline_feat_high_mean_age.sort_values("age (Ma)")
    coastline_feat_high_spread_age = coastline_feat_high_spread_age.sort_values("age (Ma)")

    # Arrays
    feat_high_vals = coastline_feat_high_mean_age[feature].values.reshape(-1, 1)
    feat_high_spread = coastline_feat_high_spread_age[feature].values.reshape(-1, 1)

    # Smooth feature mean and spread
    _, feat_high_vals_smooth = smooth_curve(
        ages_high, feat_high_vals, method="gaussian", fwhm_myr=50
    )
    _, feat_high_spread_smooth = smooth_curve(
        ages_high, feat_high_spread, method="gaussian", fwhm_myr=50
    )

    # Plot
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_facecolor("whitesmoke")

    # Histogram
    ax1.hist(
        deposit_coords_recon["age (Ma)"],
        bins=bins,
        orientation="vertical",
        color="gray",
        alpha=0.5,
        edgecolor="black"
    )
    ax1.set_xlabel("Age (Ma)", fontsize=14)
    ax1.set_ylabel("Deposit count", fontsize=14)

    # Probability axis
    ax2 = ax1.twinx()
    prob_label = (
        "Mineralisation probability"
        if probability_col == "probability"
        else "Probability adjusted by uncertainty"
    )

    ax2.plot(
        ages_high_smooth,
        probs_high_smooth,
        color="orangered",
        linewidth=2,
        label=prob_label
    )
    ax2.fill_between(
        ages_high_smooth.flatten(),
        (probs_high_smooth - probs_high_spread_smooth).flatten(),
        (probs_high_smooth + probs_high_spread_smooth).flatten(),
        color="orangered",
        alpha=0.2
    )

    ax2.set_xlim(time_max, time_min)
    ax2.set_ylim(0.5, 1)
    ax2.set_ylabel(
        "Mineralisation probability (P > 0.5)"
        if probability_col == "probability"
        else "Probability adjusted by uncertainty (P > 0.5)",
        fontsize=14
    )

    # Feature axis
    ax3 = ax1.twinx()
    ax3.spines["right"].set_position(("outward", 60))

    feature_label = format_feature_name(feature)
    uncertainty_label = "±1σ" if spread_method == "std" else "±MAD"

    ax3.plot(
        ages_high_smooth,
        feat_high_vals_smooth,
        color="royalblue",
        linewidth=2,
        label=f"{feature_label} ({uncertainty_label})"
    )
    ax3.fill_between(
        ages_high_smooth.flatten(),
        (feat_high_vals_smooth - feat_high_spread_smooth).flatten(),
        (feat_high_vals_smooth + feat_high_spread_smooth).flatten(),
        color="royalblue",
        alpha=0.2
    )

    ax3.set_ylabel(feature_label, fontsize=14)

    ax1.tick_params(labelsize=12)
    ax2.tick_params(labelsize=12)
    ax3.tick_params(labelsize=12)

    # Legend
    spread_label = "±1σ" if spread_method == "std" else "±MAD"

    lines2, labels2 = ax2.get_legend_handles_labels()
    lines3, labels3 = ax3.get_legend_handles_labels()

    labels2 = [f"{l} ({spread_label})" for l in labels2]

    lines = lines2 + lines3
    labels = labels2 + labels3
    ax3.legend(lines, labels, loc="upper left", frameon=True, fontsize=12)

    plt.tight_layout()

    clean_feature = clean_feature_name(feature)
    fig_path = os.path.join(figs_dir, f"coastline_avg_trend_high_{clean_feature}.png")
    if not os.path.exists(fig_path):
        plt.savefig(
            fig_path,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()

This cell allows the user to plot the average values of two uncertainty metrics—entropy and vote variance—alongside the average probability values.

In [ ]:
@interact
def show_uncertainty(spread_method=["std", "mad", "none"]):
    coastline_prob_high = coastline_prob[coastline_prob["probability"] > 0.5]
    # Mean entropy by age
    coastline_entropy_high_mean_age = (
        coastline_prob_high.groupby("age (Ma)")["entropy"]
        .mean()
        .reset_index()
    )

    # Mean vote variance by age
    coastline_vote_var_high_mean_age = (
        coastline_prob_high.groupby("age (Ma)")["vote_variance"]
        .mean()
        .reset_index()
    )

    # Mean probability by age
    coastline_prob_high_mean_age = (
        coastline_prob_high.groupby("age (Ma)")["probability"]
        .mean()
        .reset_index()
    )

    # Get all unique ages from original data
    all_ages = coastline_prob["age (Ma)"].unique()
    complete_ages = pd.DataFrame({"age (Ma)": all_ages})

    # Merge and fill missing values
    coastline_entropy_high_mean_age = complete_ages.merge(
        coastline_entropy_high_mean_age,
        on="age (Ma)",
        how="left"
    ).fillna(coastline_entropy_high_mean_age["entropy"].mean())

    coastline_vote_var_high_mean_age = complete_ages.merge(
        coastline_vote_var_high_mean_age,
        on="age (Ma)",
        how="left"
    ).fillna(coastline_vote_var_high_mean_age["vote_variance"].mean())

    coastline_prob_high_mean_age = complete_ages.merge(
        coastline_prob_high_mean_age,
        on="age (Ma)",
        how="left"
    ).fillna(coastline_prob_high_mean_age["probability"].mean())

    # Sort by age
    coastline_entropy_high_mean_age = coastline_entropy_high_mean_age.sort_values("age (Ma)")
    coastline_vote_var_high_mean_age = coastline_vote_var_high_mean_age.sort_values("age (Ma)")
    coastline_prob_high_mean_age = coastline_prob_high_mean_age.sort_values("age (Ma)")

    # Arrays
    ages_high = coastline_entropy_high_mean_age["age (Ma)"].values.reshape(-1, 1)
    entropies_high = coastline_entropy_high_mean_age["entropy"].values.reshape(-1, 1)
    vote_vars_high = coastline_vote_var_high_mean_age["vote_variance"].values.reshape(-1, 1)
    probs_high = coastline_prob_high_mean_age["probability"].values.reshape(-1, 1)

    # Smooth means
    ages_high_smooth, entropies_high_smooth = smooth_curve(
        ages_high, entropies_high, method="gaussian", fwhm_myr=50
    )
    _, vote_vars_high_smooth = smooth_curve(
        ages_high, vote_vars_high, method="gaussian", fwhm_myr=50
    )
    _, probs_high_smooth = smooth_curve(
        ages_high, probs_high, method="gaussian", fwhm_myr=50
    )

    # Compute and smooth spreads only if needed
    if spread_method != "none":
        coastline_entropy_high_spread_age = (
            coastline_prob_high.groupby("age (Ma)")["entropy"]
            .apply(lambda x: compute_spread(x, method=spread_method))
            .reset_index(name="entropy")
        )

        coastline_vote_var_high_spread_age = (
            coastline_prob_high.groupby("age (Ma)")["vote_variance"]
            .apply(lambda x: compute_spread(x, method=spread_method))
            .reset_index(name="vote_variance")
        )

        coastline_prob_high_spread_age = (
            coastline_prob_high.groupby("age (Ma)")["probability"]
            .apply(lambda x: compute_spread(x, method=spread_method))
            .reset_index(name="probability")
        )

        # Merge and fill missing
        coastline_entropy_high_spread_age = complete_ages.merge(
            coastline_entropy_high_spread_age,
            on="age (Ma)",
            how="left"
        ).fillna(coastline_entropy_high_spread_age["entropy"].mean())

        coastline_vote_var_high_spread_age = complete_ages.merge(
            coastline_vote_var_high_spread_age,
            on="age (Ma)",
            how="left"
        ).fillna(coastline_vote_var_high_spread_age["vote_variance"].mean())

        coastline_prob_high_spread_age = complete_ages.merge(
            coastline_prob_high_spread_age,
            on="age (Ma)",
            how="left"
        ).fillna(coastline_prob_high_spread_age["probability"].mean())

        # Sort
        coastline_entropy_high_spread_age = coastline_entropy_high_spread_age.sort_values("age (Ma)")
        coastline_vote_var_high_spread_age = coastline_vote_var_high_spread_age.sort_values("age (Ma)")
        coastline_prob_high_spread_age = coastline_prob_high_spread_age.sort_values("age (Ma)")

        # Arrays
        entropies_spread = coastline_entropy_high_spread_age["entropy"].values.reshape(-1, 1)
        vote_vars_spread = coastline_vote_var_high_spread_age["vote_variance"].values.reshape(-1, 1)
        probs_spread = coastline_prob_high_spread_age["probability"].values.reshape(-1, 1)

        # Smooth spreads
        _, entropies_spread_smooth = smooth_curve(
            ages_high, entropies_spread, method="gaussian", fwhm_myr=50
        )
        _, vote_vars_spread_smooth = smooth_curve(
            ages_high, vote_vars_spread, method="gaussian", fwhm_myr=50
        )
        _, probs_spread_smooth = smooth_curve(
            ages_high, probs_spread, method="gaussian", fwhm_myr=50
        )

    # Plot
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_facecolor("whitesmoke")

    # Histogram
    ax1.hist(
        deposit_coords_recon["age (Ma)"],
        bins=bins,
        orientation="vertical",
        color="gray",
        alpha=0.5,
        edgecolor="black"
    )
    ax1.set_xlabel("Age (Ma)", fontsize=14)
    ax1.set_ylabel("Deposit count", fontsize=14)

    # Label text
    uncertainty_label = "" if spread_method == "none" else ("±1σ" if spread_method == "std" else "±MAD")

    # Probability axis
    ax2 = ax1.twinx()
    ax2.plot(
        ages_high_smooth,
        probs_high_smooth,
        color="orangered",
        linewidth=2,
        label=f"Mineralisation probability {f'({uncertainty_label})' if uncertainty_label else ''}"
    )

    if spread_method != "none":
        ax2.fill_between(
            ages_high_smooth.flatten(),
            (probs_high_smooth - probs_spread_smooth).flatten(),
            (probs_high_smooth + probs_spread_smooth).flatten(),
            color="orangered",
            alpha=0.2
        )

    ax2.set_xlim(time_max, time_min)
    ax2.set_ylim(0.5, 1)
    ax2.set_ylabel("Mineralisation probability (P > 0.5)", fontsize=14)

    # Entropy axis
    ax3 = ax1.twinx()
    ax3.spines["right"].set_position(("outward", 60))
    ax3.plot(
        ages_high_smooth,
        entropies_high_smooth,
        color="royalblue",
        linewidth=2,
        label=f"Entropy {f'({uncertainty_label})' if uncertainty_label else ''}"
    )

    if spread_method != "none":
        ax3.fill_between(
            ages_high_smooth.flatten(),
            (entropies_high_smooth - entropies_spread_smooth).flatten(),
            (entropies_high_smooth + entropies_spread_smooth).flatten(),
            color="royalblue",
            alpha=0.2
        )

    ax3.set_ylabel("Entropy", fontsize=14)

    # Vote variance axis
    ax4 = ax1.twinx()
    ax4.spines["right"].set_position(("outward", 120))
    ax4.plot(
        ages_high_smooth,
        vote_vars_high_smooth,
        color="limegreen",
        linewidth=2,
        label=f"Vote variance {f'({uncertainty_label})' if uncertainty_label else ''}"
    )

    if spread_method != "none":
        ax4.fill_between(
            ages_high_smooth.flatten(),
            (vote_vars_high_smooth - vote_vars_spread_smooth).flatten(),
            (vote_vars_high_smooth + vote_vars_spread_smooth).flatten(),
            color="limegreen",
            alpha=0.2
        )

    ax4.set_ylabel("Vote variance", fontsize=14)

    # Legend
    lines, labels = [], []
    for ax in [ax2, ax3, ax4]:
        l, lab = ax.get_legend_handles_labels()
        lines += l
        labels += lab

    ax4.legend(lines, labels, loc="upper left", frameon=True, fontsize=12)

    plt.tight_layout()

    fig_path = os.path.join(figs_dir, "coastline_avg_unc_trend.png")
    if not os.path.exists(fig_path):
        fig.savefig(
            fig_path,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()

### Global Prospectivity Map of Porphyry Mineralisation

This code block generates global grids of probability, entropy, and vote variance for points within the present-day continental landmass by averaging their values through time. The grids are created by interpolating these point values, with the interpolation step filling gaps between measurements. We use nearest-neighbor interpolation but mask out grid cells located far from any known data points (based on grid resolution) to avoid misleading extrapolations. The final gridded datasets are saved as compressed NetCDF files.

In [ ]:
if not os.path.isfile(coastline_prob_present_day_grid_filename):
    # Create probability, entropy, and vote variance grids for the points within present-day continental landmass
    for column, filename in zip(
        [
            "probability",
            "entropy",
            "vote_variance",
            "uncertainty",
            "probability_uncertainty",
        ],
        [
            coastline_prob_present_day_grid_filename,
            coastline_entropy_present_day_grid_filename,
            coastline_vote_variance_present_day_grid_filename,
            coastline_uncertainty_present_day_grid_filename,
            coastline_prob_unc_present_day_grid_filename,
        ]
    ):
        xmin, xmax, ymin, ymax = -180, 180, -90, 90

        # Create the meshes
        grid_lons = np.arange(xmin, xmax + grid_resolution, grid_resolution)
        grid_lats = np.arange(ymin, ymax + grid_resolution, grid_resolution)
        grid_mlons, grid_mlats = np.meshgrid(grid_lons, grid_lats)

        data_lons = coastline_prob_present_day["present_lon"]
        data_lats = coastline_prob_present_day["present_lat"]
        data_values = coastline_prob_present_day[column]

        # Interpolate using the nearest neighbour method
        points = np.column_stack((data_lons, data_lats))
        arr = griddata(points, data_values, (grid_mlons, grid_mlats), method="nearest")  # or "linear", "cubic"

        # Mask distant grid nodes
        tree = cKDTree(points)
        flat_grid_points = np.column_stack((grid_mlons.ravel(), grid_mlats.ravel()))
        distances, _ = tree.query(flat_grid_points, k=1)

        mask = distances > grid_resolution
        arr.ravel()[mask] = np.nan  # Mask out distant nodes        

        # Create the dataset
        dset = xr.Dataset(
            data_vars={
                "z": (("lat", "lon"), arr),
            },
            coords={
                "lon": grid_lons,
                "lat": grid_lats,
            },
        )

        # Set projection info
        dset.rio.write_crs(4326, inplace=True)

        # Save as NetCDF
        dset.to_netcdf(
            filename,
            encoding={
                "z": {
                    "zlib": True,
                    "dtype": "float32",
                }
            },
        )

This cell generates an interactive probability map that displays mineralisation probability, entropy, and vote variance for points within the present-day continental landmasses, alongside known mineral occurrences. It also plots seafloor age and plate boundaries, including mid-ocean ridges and transform faults. The visualisation highlights spatial patterns—showing how mineralisation probability relates to tectonic features—and temporal evolution, illustrating how favourable conditions and identified potential areas align with known mineral occurrences.

In [ ]:
projection = ccrs.Mollweide(central_longitude=30)
deposit_coords_recon_all_filtered = pd.read_csv(deposit_coords_recon_all_filtered_filename)
time = 0

gplot.time = time

deposit_coords_recon_t = deposit_coords_recon_all_filtered[[f"lon_{time}", f"lat_{time}", "weight"]]
deposit_coords_recon_t = deposit_coords_recon_t.dropna()
    
agegrid_filename =  f"seafloor_age_{time}Ma.nc"
agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
agegrid = gplately.grids.read_netcdf_grid(agegrid_file)

map_types = ["Probability", "Entropy", "Vote Variance", "Uncertainty", "Adjusted Probability"]

@interact
def show_map(map_type=map_types):
    # Load the appropriate raster data based on the selected map type
    if map_type==map_types[0]:
        raster = Raster(coastline_prob_present_day_grid_filename)
        raster.data *= 100.0
    elif map_type==map_types[1]:
        raster = Raster(coastline_entropy_present_day_grid_filename)
    elif map_type==map_types[2]:
        raster = Raster(coastline_vote_variance_present_day_grid_filename)
    elif map_type==map_types[3]:
        raster = Raster(coastline_uncertainty_present_day_grid_filename)
    elif map_type==map_types[4]:
        raster = Raster(coastline_prob_unc_present_day_grid_filename)

    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_axes(
        [0.1, 0.1, 0.8, 0.8],
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba("darkgray", alpha=0.5)),
    )

    # Create axes for colourbars
    cax_grd = fig.add_axes([0.33, 0.17, 0.25, 0.02])
    cax_bg = fig.add_axes([0.62, 0.17, 0.25, 0.02])

    # Plot seafloor age as background
    bg = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)
    # Plot continental areas in dark grey
    gplot.plot_coastlines(ax, facecolor="darkgray", edgecolor="none", zorder=2)
    # Plot plate motion vectors (arrows showing plate movement direction)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # Plot the raster data based on the selected map type
    # This will display the mineralisation probability, entropy, or vote variance
    if map_type==map_types[0]:
        # grd = raster.imshow(ax=ax, cmap=ccm.hawaii_r, vmin=0, vmax=100, alpha=0.9, zorder=4)
        grd = raster.imshow(ax=ax, cmap=ccm.hawaii_r, vmin=50, vmax=100, alpha=0.9, zorder=4)
    elif map_type==map_types[1] or map_type==map_types[3] or map_type==map_types[4]:
        grd = raster.imshow(ax=ax, cmap=ccm.hawaii_r, vmin=0, vmax=1, alpha=0.9, zorder=4)
    elif map_type==map_types[2]:
        grd = raster.imshow(ax=ax, cmap=ccm.hawaii_r, vmin=0, vmax=0.25, alpha=0.9, zorder=4)

    gplot.plot_topological_plate_boundaries(ax, color="dimgray", linewidth=1.2, zorder=5)
    gplot.plot_trenches(ax, color="k", alpha=0.3, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color="k", alpha=0.3, zorder=7)

    # Plot reconstructed mineral occurrences
    sc1 = ax.scatter(
        deposit_coords_recon_t[f"lon_{time}"],
        deposit_coords_recon_t[f"lat_{time}"],
        transform=ccrs.PlateCarree(),
        marker="o",
        facecolor="yellow",
        edgecolor="black",
        s = [w * 10 for w in deposit_coords_recon_t["weight"]],
        alpha=0.3,
        zorder=8
    )

    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=9)

    # If central longitude 60 deg
    # ax.text(0.49,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    # ax.text(0.46,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    # ax.text(0.40,-0.025, "60°W", transform=ax.transAxes, fontsize=16)
    # If central longitude 30 deg
    ax.text(0.505,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.48,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.425,-0.027, "60°W", transform=ax.transAxes, fontsize=16)

    gl.top_labels=False
    gl.bottom_labels=False

    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}

    # Add a colourbar for the mineralisation probability, entropy, or vote variance
    cbar_grd = fig.colorbar(grd, cax=cax_grd, orientation="horizontal")
    if map_type==map_types[0]:
        cbar_grd.set_label("Mineralisation Probability", fontsize=16, labelpad=10)
    elif map_type==map_types[1]:
        cbar_grd.set_label("Entropy", fontsize=16, labelpad=10)
    elif map_type==map_types[2]:
        cbar_grd.set_label("Vote Variance", fontsize=16, labelpad=10)
    elif map_type==map_types[3]:
        cbar_grd.set_label("Uncertainty", fontsize=16, labelpad=10)
    elif map_type==map_types[4]:
        cbar_grd.set_label("Adjusted Probability", fontsize=16, labelpad=10)
    cbar_grd.ax.tick_params(labelsize=16)

    # Add a colourbar for seafloor age
    cbar_bg = fig.colorbar(bg, cax=cax_bg, orientation="horizontal", extend="max")
    cbar_bg.set_label("Seafloor Age (Ma)", fontsize=16, labelpad=10)
    cbar_bg.set_ticks([0, 50, 100, 150, 200])
    cbar_bg.ax.tick_params(labelsize=16)

    # Dummy handle to trigger custom handler
    trench_handle = Line2D([], [], color="black")
    
    # Add custom handles
    custom_handles = [
        Patch(facecolor="darkgray", edgecolor="none"),
        trench_handle,
        Line2D([0], [0], color="dimgray", lw=1.2),
        Line2D([0], [0], marker="o", markerfacecolor="yellow", markeredgecolor="black", markersize=15, linestyle="None"),
    ]
    custom_labels = [
        "Continental crust",
        "Trench line",
        "Plate boundary",
        "Mineral occurrence",
    ]

    # Draw legend
    ax.legend(custom_handles, custom_labels, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.22),
              handler_map={trench_handle: HandlerTrenchLine()})

    plt.show()

In [ ]:
deposit_coords_recon_all_filtered = pd.read_csv(deposit_coords_recon_all_filtered_filename)
time = 0

gplot.time = time

deposit_coords_recon_t = deposit_coords_recon_all_filtered[
    [f"lon_{time}", f"lat_{time}", "weight"]
].dropna()

agegrid_filename = f"seafloor_age_{time}Ma.nc"
agegrid_file = os.path.join(agegrid_dir, agegrid_filename)
agegrid = gplately.grids.read_netcdf_grid(agegrid_file)

# Keep only these map types
map_types = ["Probability", "Uncertainty", "Adjusted Probability"]

CLASS_LABELS = ["Very low", "Low", "Medium", "High", "Very high"]

def get_raster_and_label(map_type):
    if map_type == "Probability":
        raster = Raster(coastline_prob_present_day_grid_filename)
        raster.data = raster.data * 100.0
        label = "Mineralisation Probability"
    elif map_type == "Uncertainty":
        raster = Raster(coastline_uncertainty_present_day_grid_filename)
        label = "Uncertainty"
    elif map_type == "Adjusted Probability":
        raster = Raster(coastline_prob_unc_present_day_grid_filename)
        raster.data = raster.data * 100.0
        label = "Adjusted Probability by Uncertainty"
    else:
        raise ValueError(f"Unknown map type: {map_type}")
    return raster, label

def quantile_breaks(data, n_classes=5):
    """
    Compute quantile-based class edges for skewed raster data.
    Falls back to equal interval if quantiles collapse.
    """
    values = np.asarray(data, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        raise ValueError("Raster contains no finite values.")

    probs = np.linspace(0, 1, n_classes + 1)
    bins = np.quantile(values, probs)
    bins = np.asarray(bins, dtype=float)

    # If skew / repeated values causes duplicate bin edges, fall back
    if np.unique(bins).size < (n_classes + 1):
        bins = np.linspace(values.min(), values.max(), n_classes + 1)

    return bins

def discrete_cmap_and_norm(bins):
    n_intervals = len(bins) - 1
    base_cmap = ccm.hawaii_r
    # base_cmap = ccm.batlow
    colors = base_cmap(np.linspace(0.15, 0.95, n_intervals))
    cmap = mcolors.ListedColormap(colors)
    norm = mcolors.BoundaryNorm(bins, cmap.N, clip=True)
    return cmap, norm

@interact
def show_map(map_type=map_types):
    raster, raster_label = get_raster_and_label(map_type)

    raster_values = np.asarray(raster.data, dtype=float)
    bins = quantile_breaks(raster_values, n_classes=5)

    cmap, norm = discrete_cmap_and_norm(bins)

    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_axes(
        [0.1, 0.1, 0.8, 0.8],
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba("darkgray", alpha=0.5)),
    )

    # Move colourbars lower
    cax_grd = fig.add_axes([0.33, 0.14, 0.25, 0.035])
    cax_bg  = fig.add_axes([0.62, 0.17, 0.25, 0.02])

    # Background seafloor age
    bg = gplot.plot_grid(
        ax,
        agegrid.data,
        cmap=ccm.lapaz_r,
        vmin=0,
        vmax=230,
        alpha=0.7,
        zorder=1,
    )

    # Context layers
    gplot.plot_coastlines(ax, facecolor="darkgray", edgecolor="none", zorder=2)
    gplot.plot_plate_motion_vectors(
        ax,
        spacingX=10,
        spacingY=10,
        normalise=True,
        alpha=0.1,
        zorder=3,
    )

    # Classified raster
    grd = raster.imshow(
        ax=ax,
        cmap=cmap,
        norm=norm,
        alpha=0.9,
        zorder=4,
    )

    gplot.plot_topological_plate_boundaries(ax, color="dimgray", linewidth=1.2, zorder=5)
    gplot.plot_trenches(ax, color="k", alpha=0.3, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color="k", alpha=0.3, zorder=7)

    # Mineral occurrences
    ax.scatter(
        deposit_coords_recon_t[f"lon_{time}"],
        deposit_coords_recon_t[f"lat_{time}"],
        transform=ccrs.PlateCarree(),
        marker="o",
        facecolor="yellow",
        edgecolor="black",
        s=[w * 10 for w in deposit_coords_recon_t["weight"]],
        alpha=0.3,
        zorder=8,
    )

    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        x_inline=False,
        linewidth=1,
        color="gray",
        alpha=0.3,
        linestyle="--",
        zorder=9,
    )

    # If central longitude 60 deg
    # ax.text(0.49,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    # ax.text(0.46,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    # ax.text(0.40,-0.025, "60°W", transform=ax.transAxes, fontsize=16)
    # If central longitude 30 deg
    ax.text(0.505,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.48,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.425,-0.027, "60°W", transform=ax.transAxes, fontsize=16)

    gl.top_labels = False
    gl.bottom_labels = False
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}

    # Classified colourbar
    tick_positions = 0.5 * (bins[:-1] + bins[1:])

    cbar_grd = fig.colorbar(
        grd,
        cax=cax_grd,
        orientation="horizontal",
        boundaries=bins,
        ticks=tick_positions,
        spacing="uniform",   # equal-size boxes
    )

    # Put category labels under the bar
    cbar_grd.ax.set_xticklabels(CLASS_LABELS, fontsize=14)

    # Put title above the bar
    cbar_grd.ax.xaxis.set_label_position("top")
    cbar_grd.set_label(raster_label, fontsize=16, labelpad=8)

    # Remove tick marks for a cleaner classified legend
    cbar_grd.ax.tick_params(axis="x", length=0, pad=6)

    cbar_bg = fig.colorbar(
        bg,
        cax=cax_bg,
        orientation="horizontal",
        extend="max",
    )
    cbar_bg.set_label("Seafloor Age (Ma)", fontsize=16, labelpad=10)
    cbar_bg.set_ticks([0, 50, 100, 150, 200])
    cbar_bg.ax.tick_params(labelsize=16)

    # Legend
    trench_handle = Line2D([], [], color="black")
    custom_handles = [
        Patch(facecolor="darkgray", edgecolor="none"),
        trench_handle,
        Line2D([0], [0], color="dimgray", lw=1.2),
        Line2D(
            [0], [0],
            marker="o",
            markerfacecolor="yellow",
            markeredgecolor="black",
            markersize=15,
            linestyle="None",
        ),
    ]
    custom_labels = [
        "Continental crust",
        "Trench line",
        "Plate boundary",
        "Mineral occurrence",
    ]

    ax.legend(
        custom_handles,
        custom_labels,
        fontsize=16,
        loc="lower left",
        bbox_to_anchor=(0, -0.23),
        handler_map={trench_handle: HandlerTrenchLine()},
    )

    fig_path = os.path.join(figs_dir, "probability_uncertainty.png")
    if not os.path.exists(fig_path):
        fig.savefig(
            fig_path,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()

### Prediction-Area Plot

This code segment generates a Prediction–Area (P–A) plot, a valuable tool for assessing the effectiveness of the machine learning model in predicting porphyry mineralisation. The P–A plot quantitatively compares the spatial distribution of predicted probabilities with the known locations of mineral occurrences.

For each known mineral occurrence, the corresponding probability value is extracted from the raster grid by selecting the nearest grid cell to the occurrence's geographic coordinates. This provides insights into the model's precision and efficiency in targeting exploration areas.

The analysis considers a range of probability thresholds from 0 to 1. For each threshold, two key percentages are computed:

- **Occurrence Percentage**: The proportion of known mineral occurrences with predicted probabilities greater than or equal to the threshold. This measures how many actual deposits are captured by a given probability cutoff.
- **Area Percentage**: The proportion of the study area with predicted probabilities exceeding the same threshold. This represents how much geographic area would need to be explored if the threshold were applied as a filter.

The resulting curve illustrates model performance by showing how increasing the probability cutoff reduces the exploration area but may also exclude some known occurrences.

In [ ]:
# Load prospectivity data
prospectivity_data = xr.open_dataset(coastline_prob_present_day_grid_filename)
prospectivity_data = prospectivity_data["z"].squeeze()

# Extract probability values at known mineral occurrence locations
occurrence_probabilities = []
for _, row in deposit_coords_recon.iterrows():
    lon, lat = row["present_lon"], row["present_lat"]
    # Extract probability value at this point
    prob_value = prospectivity_data.sel(lon=lon, lat=lat, method="nearest").item()
    occurrence_probabilities.append(prob_value)

# Convert to numpy array for easier manipulation
occurrence_probabilities = np.array(occurrence_probabilities)
occurrence_probabilities = occurrence_probabilities[~np.isnan(occurrence_probabilities)]

# Calculate total study area and number of valid cells
total_area = np.count_nonzero(~np.isnan(prospectivity_data))
total_occurrences = len(occurrence_probabilities)

# Define the probability thresholds
thresholds = np.linspace(0, 1, 100)

# Initialise lists to store results
occurrence_percentages = []
area_percentages = []

# Calculate percentages for each threshold
for threshold in thresholds:
    occurrence_percentage = np.sum(occurrence_probabilities >= threshold) / total_occurrences * 100
    area_percentage = np.sum(prospectivity_data >= threshold) / total_area * 100
    occurrence_percentages.append(occurrence_percentage)
    area_percentages.append(area_percentage)

# Convert lists to numpy arrays for plotting
occurrence_percentages = np.array(occurrence_percentages)
area_percentages = np.array(area_percentages)

# Set the font sizes
SMALL_SIZE = 12
MEDIUM_SIZE = 14
LARGE_SIZE = 16

# Create the P-A plot
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.set_facecolor("whitesmoke")

# Plot the occurrence percentages
ax1.plot(thresholds, occurrence_percentages, color="royalblue", linewidth=2)
ax1.set_xlabel("Probability", fontsize=MEDIUM_SIZE)
ax1.set_ylabel("Percentage of Known Mineral Occurrences", color="royalblue", fontsize=MEDIUM_SIZE)
ax1.set_ylim(0, 100)
ax1.set_xlim(0.5, 1)
ax1.grid(True)

# Set tick label sizes
ax1.tick_params(axis="both", labelsize=SMALL_SIZE)

# Create a second y-axis to plot the area percentages
ax2 = ax1.twinx()
ax2.plot(thresholds, area_percentages, color="orangered", linewidth=2)
ax2.set_ylabel("Percentage of Study Area", color="orangered", fontsize=MEDIUM_SIZE)
ax2.set_ylim(100, 0)  # Inverted y-axis
ax2.tick_params(axis="y", labelsize=SMALL_SIZE)

intersection_x = 0.76
intersection_y1 = 73.5
intersection_y2 = 26.5

h_line = ax1.hlines(y=intersection_y1, xmin=0, xmax=1, color="g", linestyles=":")
v_line = ax1.vlines(x=intersection_x, ymin=0, ymax=100, color="g", linestyles=":")

# Add a text box to explain the intersection point
ax1.annotate(f"At probability {intersection_x}:\n{intersection_y1}% of occurrences\n{intersection_y2}% of area",
             xy=(0.25, 0.8), xycoords="axes fraction",
             ha="center", va="top",
             fontsize=SMALL_SIZE,
             bbox=dict(boxstyle="round", fc="white", ec="gray", alpha=0.8))

# plt.title("Prediction-Area Plot")
plt.tight_layout()  # Adjust layout to prevent clipping of labels

# Adjust the plot margins to make room for annotations
plt.subplots_adjust(left=0.15, right=0.85)

fig_path = os.path.join(figs_dir, "pa.png")
if not os.path.exists(fig_path):
    fig.savefig(
        fig_path,
        dpi=300,
        bbox_inches="tight",
    )

plt.show()